# 06 — Modèles classiques NLP · Databricks
**Exécuter après** : `05_baseline_tfidf_db.ipynb`
---

In [0]:
%pip install scipy scikit-learn joblib xgboost vaderSentiment seaborn -q

In [0]:
import sys, os, json, time
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz, hstack
import scipy.sparse as sp

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

plt.style.use('dark_background')

# Chemins Volume Unity Catalog
# Trouver le chemin exact : Databricks > Data > Volumes > clic droit sur "fakenews" > Copy path
VOLUME        = '/Volumes/workspace/default/fakenews'   # <- MODIFIER catalog/schema si besoin

COLAB_DIR     = VOLUME + '/colab_exports'
MODELS_DIR    = VOLUME + '/models'
REPORTS_DIR   = VOLUME + '/reports'
for d in [MODELS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)
print('Imports OK')

## Section 1 — Chargement et enrichissement des features

In [0]:
X_train_tfidf = load_npz(os.path.join(COLAB_DIR, 'train_features.npz'))
X_test_tfidf  = load_npz(os.path.join(COLAB_DIR, 'test_features.npz'))
y_train = np.load(os.path.join(COLAB_DIR, 'train_labels.npy'))
y_test  = np.load(os.path.join(COLAB_DIR, 'test_labels.npy'))

train_texts = pd.read_csv(os.path.join(COLAB_DIR, 'train_texts.csv'))
test_texts  = pd.read_csv(os.path.join(COLAB_DIR, 'test_texts.csv'))

print(f'TF-IDF train : {X_train_tfidf.shape}')

In [0]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def compute_nlp_features(texts):
    features = []
    for text in texts:
        if not isinstance(text, str):
            features.append([0, 0.0, 0.0, 0, 0])
            continue
        word_count    = len(text.split())
        sentiment     = analyzer.polarity_scores(text)['compound']
        alpha         = [c for c in text if c.isalpha()]
        capital_ratio = sum(1 for c in alpha if c.isupper()) / len(alpha) if alpha else 0.0
        features.append([word_count, sentiment, capital_ratio, text.count('!'), text.count('?')])
    return np.array(features)

print('Calcul des features NLP (peut prendre quelques minutes)...')
nlp_train = compute_nlp_features(train_texts['clean_text'])
nlp_test  = compute_nlp_features(test_texts['clean_text'])

X_train = hstack([X_train_tfidf, sp.csr_matrix(nlp_train)])
X_test  = hstack([X_test_tfidf,  sp.csr_matrix(nlp_test)])
print(f'Features combinees : {X_train.shape}')

## Section 2 — Comparaison des modèles

In [0]:
MODELS = {
    'LogReg_NLP':   LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42),
    'XGBoost':      XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42),
}

results = []
trained_models = {}

for name, model in MODELS.items():
    print(f'\nEntrainement : {name}...')
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba)
    results.append({'model': name, 'accuracy': acc, 'f1_score': f1, 'auc_roc': auc, 'train_time_s': round(elapsed, 1)})
    trained_models[name] = model
    print(f'  {name} | Accuracy={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f} | {elapsed:.1f}s')

results_df = pd.DataFrame(results).sort_values('f1_score', ascending=False)
print('\n=== COMPARAISON ===')
display(results_df)

## Section 3 — Sauvegarde du meilleur modèle

In [0]:
best_row   = results_df.iloc[0]
best_name  = best_row['model']
best_model = trained_models[best_name]

baseline_dir = os.path.join(MODELS_DIR, 'baseline')
os.makedirs(baseline_dir, exist_ok=True)
best_path = os.path.join(baseline_dir, f'best_classical_{best_name}.pkl')
joblib.dump(best_model, best_path)
print(f'Meilleur modele sauvegarde : {best_path}')

metadata_path = os.path.join(MODELS_DIR, 'metadata.json')
metadata = {}
if os.path.exists(metadata_path):
    with open(metadata_path) as f:
        metadata = json.load(f)

for row in results_df.to_dict('records'):
    metadata[row['model']] = {
        'accuracy': round(row['accuracy'], 4), 'f1_score': round(row['f1_score'], 4),
        'auc_roc': round(row['auc_roc'], 4), 'trained_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Meilleur modele : {best_name} (F1={best_row["f1_score"]:.4f})')
print('Prochaine etape : 07_evaluation_db.ipynb')